In [2]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, precision_recall_curve,
    auc, confusion_matrix, ConfusionMatrixDisplay,
)

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except Exception as exc:
    HAS_XGBOOST = False
    print("XGBoost unavailable:", exc)

RANDOM_STATE = 42

possible_folders = [Path.cwd(), Path.cwd() / "Data Science", Path.home() / "Desktop" / "Data Science"]
project_folder = next(
    (folder.resolve() for folder in possible_folders if (folder / "Fire2Air_model_ready_checkpoint.csv").exists()),
    None,
)
if project_folder is None:
    raise FileNotFoundError("Fire2Air_model_ready_checkpoint.csv not found. Run Thi's notebook first.")

output_root = project_folder / "outputs_prt661"
table_dir = output_root / "tables"
figure_dir = output_root / "figures"
model_dir = output_root / "models"
for folder in [table_dir, figure_dir, model_dir]:
    folder.mkdir(parents=True, exist_ok=True)

integrated_daily = pd.read_csv(
    project_folder / "Fire2Air_model_ready_checkpoint.csv",
    parse_dates=["date", "target_date"],
)

manifest_path = project_folder / "Fire2Air_feature_manifest.json"
with open(manifest_path, "r", encoding="utf-8") as f:
    manifest = json.load(f)
model_features = manifest["features"]
classification_target = manifest["targets"]["classification"]

print("Project folder:", project_folder)
print("Dataset shape:", integrated_daily.shape)
print("Shared predictors:", len(model_features))

XGBoost unavailable: No module named 'xgboost'
Project folder: C:\Users\dlihi\Documents\PRT661 Data Science\Assessment 2\PRT661---DATA-SCIENCE-PRACTICE---Dan5---Theme2\Source code
Dataset shape: (7530, 58)
Shared predictors: 22


## A Chronological Split & Class Balance
Training = target years 2018–2022, validation = target year 2023, final test = target year 2024.

In [3]:
classification_data = integrated_daily.dropna(
    subset=[classification_target, "target_date"]
).copy()
classification_data[classification_target] = classification_data[classification_target].astype(int)
classification_data["target_year"] = classification_data["target_date"].dt.year

train_cls = classification_data[classification_data["target_year"] <= 2022].copy()
validation_cls = classification_data[classification_data["target_year"] == 2023].copy()
test_cls = classification_data[classification_data["target_year"] == 2024].copy()

if min(len(train_cls), len(validation_cls), len(test_cls)) == 0:
    raise ValueError(
        f"Insufficient chronological data: train={len(train_cls)}, "
        f"validation={len(validation_cls)}, test={len(test_cls)}"
    )

assert train_cls["target_date"].max() < validation_cls["target_date"].min()
assert validation_cls["target_date"].max() < test_cls["target_date"].min()

balance_rows = []
for split_name, data in [("Train", train_cls), ("Validation", validation_cls), ("Test", test_cls)]:
    positives = int(data[classification_target].sum())
    balance_rows.append({
        "Split": split_name,
        "Rows": len(data),
        "Positive_Days": positives,
        "Positive_Rate": positives / len(data),
    })
class_balance = pd.DataFrame(balance_rows)
print("\nClass balance:")
print(class_balance.round(4).to_string(index=False))
class_balance.to_csv(table_dir / "model1_class_balance.csv", index=False)

X_train = train_cls[model_features]
y_train = train_cls[classification_target]
X_validation = validation_cls[model_features]
y_validation = validation_cls[classification_target]
X_test = test_cls[model_features]
y_test = test_cls[classification_target]


Class balance:
     Split  Rows  Positive_Days  Positive_Rate
     Train  4546            217         0.0477
Validation   946            100         0.1057
      Test   863             37         0.0429
